In [17]:
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile as tiff
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import ListedColormap, Normalize
from IPython.display import display
from skimage.measure import regionprops, label as cc_label
from skimage.morphology import binary_erosion, remove_small_holes

from tqdm.auto import tqdm

# =============================
# Constants
# =============================
PIXEL_SIZE_NM = 1.751
ERODE_PIXELS = 1
TARGET_STACK_SHAPE = (256, 256)

GLOBAL_VMIN = 0.0037
GLOBAL_VMAX = 0.6
NORM_15N = Normalize(vmin=GLOBAL_VMIN, vmax=GLOBAL_VMAX)

DISPLAY_NORM = Normalize(vmin=0.0, vmax=3.0)

SHOW_PLOTS = False

EXPECTED_CHANNELS_7 = [
    "16O",
    "12C21H",
    "12C14N",
    "12C15N",
    "29Si",
    "31P",
    "32S",
]

EXPECTED_CHANNELS_6 = [
    "16O",
    "12C21H",
    "12C14N",
    "12C15N",
    "31P",
    "32S",
]

# Illustrator-friendly PDF text
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["savefig.facecolor"] = "white"
mpl.rcParams["savefig.transparent"] = False

# =============================
# Root paths
# =============================
INPUT_ROOT = Path("input")
OUTPUT_ROOT = Path("output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if not INPUT_ROOT.exists():
    raise FileNotFoundError(f"Input root not found: {INPUT_ROOT.resolve()}")

# =============================
# NPZ output (shared across samples)
# =============================
NPZ_ROOT = Path.cwd() / "output_npz"
NPZ_ROOT.mkdir(parents=True, exist_ok=True)


# =============================
# LUT loading
# =============================
def load_fiji_lut_try(paths=(Path("LUT.csv"), Path.cwd() / "LUT.csv", Path("/mnt/data/LUT.csv"))):
    for p in paths:
        try:
            if not p.exists():
                continue

            df = pd.read_csv(p)
            cols = [c.strip().lower() for c in df.columns]

            if "red" in cols and "green" in cols and "blue" in cols:
                r_idx = cols.index("red")
                g_idx = cols.index("green")
                b_idx = cols.index("blue")
            elif "r" in cols and "g" in cols and "b" in cols:
                r_idx = cols.index("r")
                g_idx = cols.index("g")
                b_idx = cols.index("b")
            else:
                r_idx, g_idx, b_idx = 1, 2, 3

            rgb = df.iloc[:, [r_idx, g_idx, b_idx]].to_numpy(dtype=float) / 255.0
            rgb = np.clip(rgb, 0.0, 1.0)
            cmap = ListedColormap(rgb, name="fiji_lut_from_csv")

            try:
                mpl.colormaps.register(cmap, name="fiji_lut_from_csv", force=True)
            except Exception:
                try:
                    mpl.cm.register_cmap(name="fiji_lut_from_csv", cmap=cmap)
                except Exception:
                    pass

            if hasattr(cmap, "with_extremes"):
                cmap = cmap.with_extremes(bad=(0, 0, 0, 0))

            print(f"Loaded LUT from {p} ({rgb.shape[0]} entries).")
            return cmap

        except Exception as e:
            print(f"Failed to load LUT from {p}: {e}")

    return None


CUSTOM_15N_CMAP = load_fiji_lut_try()
if CUSTOM_15N_CMAP is None:
    CUSTOM_15N_CMAP = plt.get_cmap("viridis")
    print("No LUT found; using viridis for 15N display.")

# =============================
# Helpers
# =============================

def whole_field_mean_norm(stack, channel_names):
    """
    Mean-normalize each channel using all finite pixels in the full image.
    """
    n_channels = stack.shape[0]
    if len(channel_names) != n_channels:
        raise ValueError(
            f"Channel-name count ({len(channel_names)}) does not match stack channels ({n_channels})."
        )

    means = np.empty(n_channels, dtype=float)
    for ch, name in enumerate(channel_names):
        vals = stack[ch]
        finite_vals = vals[np.isfinite(vals)]
        if finite_vals.size == 0:
            print(f"Warning: channel {name} has no finite pixels. Using normalization factor 1.0.")
            means[ch] = 1.0
        else:
            mu = float(np.mean(finite_vals))
            if not np.isfinite(mu) or mu == 0:
                print(f"Warning: channel {name} has invalid mean {mu}. Using normalization factor 1.0.")
                means[ch] = 1.0
            else:
                means[ch] = mu

    return stack / means[:, None, None], means

def save_label_data_npz(
    npz_root,
    label_stem,
    lab,
    sample,
    source_label_file,
    strict_bbox,
    original_bbox,
    stack,
    stack_mean_norm,
    summed,
    fractional_15N,
    em_img,
    strict_mask_crop,
    channel_names,
    field_means,
    cell_label_id=None,
    source_cell_file=None,
):
    """
    Save reconstructable data for one label as a compressed NPZ.
    """

    sy0, sy1, sx0, sx1 = strict_bbox
    oy0, oy1, ox0, ox1 = original_bbox

    raw_stack_crop = stack[:, sy0:sy1, sx0:sx1]
    mean_norm_crop = stack_mean_norm[:, sy0:sy1, sx0:sx1]
    frac_crop = fractional_15N[sy0:sy1, sx0:sx1]
    sum_crop = summed[sy0:sy1, sx0:sx1]
    em_crop = em_img[oy0:oy1, ox0:ox1]

    raw_masked = np.full_like(raw_stack_crop, np.nan, dtype=np.float32)
    mean_norm_masked = np.full_like(mean_norm_crop, np.nan, dtype=np.float32)

    for ch in range(raw_stack_crop.shape[0]):
        raw_masked[ch][strict_mask_crop] = raw_stack_crop[ch][strict_mask_crop]
        mean_norm_masked[ch][strict_mask_crop] = mean_norm_crop[ch][strict_mask_crop]

    frac_masked = np.full_like(frac_crop, np.nan, dtype=np.float32)
    frac_masked[strict_mask_crop] = frac_crop[strict_mask_crop]

    sum_masked = np.full_like(sum_crop, np.nan, dtype=np.float32)
    sum_masked[strict_mask_crop] = sum_crop[strict_mask_crop]

    npz_path = npz_root / f"{sample}_{label_stem}_label{lab:03d}.npz"

    np.savez_compressed(
        npz_path,
        sample=sample,
        label_id=lab,
        source_label_file=source_label_file,
        source_cell_file=source_cell_file if source_cell_file is not None else "",
        cell_label_id=-1 if cell_label_id is None else int(cell_label_id),
        bbox_strict=np.array([sy0, sy1, sx0, sx1], dtype=np.int32),
        bbox_original=np.array([oy0, oy1, ox0, ox1], dtype=np.int32),
        channel_names=np.array(channel_names, dtype=object),
        field_means=np.array(field_means, dtype=np.float32),
        raw_channels=raw_masked,
        mean_norm_channels=mean_norm_masked,
        fractional_15N=frac_masked,
        sum_12C14N_12C15N=sum_masked,
        em=em_crop.astype(np.float32),
    )

    return npz_path

def clean_cell_labels(cell_img):
    """
    For each positive label:
      1) fill holes
      2) keep only the largest connected component
      3) preserve the original label value
    """
    cell_img = np.rint(cell_img).astype(np.int32)
    cleaned = np.zeros_like(cell_img, dtype=np.int32)

    for lab in np.unique(cell_img):
        lab = int(lab)
        if lab <= 0:
            continue

        mask = cell_img == lab
        if not np.any(mask):
            continue

        mask = remove_small_holes(mask, area_threshold=mask.size)

        cc = cc_label(mask, connectivity=2)
        if cc.max() == 0:
            continue

        counts = np.bincount(cc.ravel())
        counts[0] = 0
        keep = int(np.argmax(counts))
        cleaned[cc == keep] = lab

    return cleaned


def load_cell_labels(cell_path: Path):
    """
    Load a cell-label TIFF and clean it.
    Binary masks are converted to connected components first.
    """
    cell_img = tiff.imread(str(cell_path))
    if cell_img.ndim != 2:
        raise ValueError(f"Expected a 2D cell label TIFF, got shape {cell_img.shape} for {cell_path.name}")

    cell_img = np.rint(cell_img).astype(np.int32)

    if np.max(cell_img) <= 1:
        print(f"Detected binary cell mask in {cell_path.name}; converting connected components to labels.")
        cell_img = cc_label(cell_img > 0, connectivity=2).astype(np.int32)

    return clean_cell_labels(cell_img)


def whole_field_mean_norm_masked(stack, channel_names, mask):
    """
    Mean-normalize each channel using only pixels inside `mask`.
    """
    n_channels = stack.shape[0]
    if len(channel_names) != n_channels:
        raise ValueError(
            f"Channel-name count ({len(channel_names)}) does not match stack channels ({n_channels})."
        )

    mask = np.asarray(mask, dtype=bool)
    if mask.shape != stack.shape[1:]:
        raise ValueError(f"Mask shape {mask.shape} does not match stack spatial shape {stack.shape[1:]}.")

    means = np.empty(n_channels, dtype=float)
    for ch, name in enumerate(channel_names):
        vals = stack[ch][mask]
        finite_vals = vals[np.isfinite(vals)]
        if finite_vals.size == 0:
            print(f"Warning: cell-normalization channel {name} has no finite pixels. Using 1.0.")
            means[ch] = 1.0
        else:
            mu = float(np.mean(finite_vals))
            if not np.isfinite(mu) or mu == 0:
                print(f"Warning: cell-normalization channel {name} has invalid mean {mu}. Using 1.0.")
                means[ch] = 1.0
            else:
                means[ch] = mu

    return stack / means[:, None, None], means


def infer_channel_names(n_channels):
    """
    Return the expected channel-name list for the TIFF stack.

    Supported:
      - 7-channel stacks: full channel list
      - 6-channel stacks: assumes 29Si is missing
    """
    if n_channels == 7:
        return EXPECTED_CHANNELS_7.copy()
    if n_channels == 6:
        print("Detected 6-channel TIFF; assuming '29Si' is the missing channel.")
        return EXPECTED_CHANNELS_6.copy()
    raise ValueError(f"Unsupported channel count: {n_channels}. Expected 6 or 7 channels.")


def strict_reduce_mask_to_shape(original_mask, out_shape):
    """
    Keep a reduced pixel only if ALL source pixels in that block are True.
    """
    in_h, in_w = original_mask.shape
    out_h, out_w = out_shape
    y_edges = np.linspace(0, in_h, out_h + 1).astype(int)
    x_edges = np.linspace(0, in_w, out_w + 1).astype(int)

    out = np.zeros((out_h, out_w), dtype=bool)

    for i in range(out_h):
        y0, y1 = y_edges[i], y_edges[i + 1]
        if y1 <= y0:
            y1 = min(y0 + 1, in_h)
        for j in range(out_w):
            x0, x1 = x_edges[j], x_edges[j + 1]
            if x1 <= x0:
                x1 = min(x0 + 1, in_w)

            block = original_mask[y0:y1, x0:x1]
            out[i, j] = bool(block.size > 0 and np.all(block))

    return out


def expand_reduced_mask_to_fullres(reduced_mask, full_shape):
    """
    Back-project a reduced mask to full resolution using the same bin edges.
    """
    full_h, full_w = full_shape
    red_h, red_w = reduced_mask.shape
    y_edges = np.linspace(0, full_h, red_h + 1).astype(int)
    x_edges = np.linspace(0, full_w, red_w + 1).astype(int)

    out = np.zeros((full_h, full_w), dtype=bool)

    for i in range(red_h):
        y0, y1 = y_edges[i], y_edges[i + 1]
        if y1 <= y0:
            y1 = min(y0 + 1, full_h)
        for j in range(red_w):
            if not reduced_mask[i, j]:
                continue
            x0, x1 = x_edges[j], x_edges[j + 1]
            if x1 <= x0:
                x1 = min(x0 + 1, full_w)
            out[y0:y1, x0:x1] = True

    return out


def bbox_of_mask(mask):
    coords = np.argwhere(mask)
    if coords.size == 0:
        return None
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    return y0, y1, x0, x1


def crop_to_bbox(arr, bbox):
    y0, y1, x0, x1 = bbox
    return arr[y0:y1, x0:x1]


def show_square_image(ax, img, title="", cmap=None, norm=None):
    ax.imshow(img, cmap=cmap, norm=norm, aspect="equal", interpolation="nearest")
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.axis("off")


def finite_values(values):
    values = np.asarray(values, dtype=float).ravel()
    return values[np.isfinite(values)]


def summarize(values):
    vals = finite_values(values)
    if vals.size == 0:
        return {
            "n": 0,
            "mean": np.nan,
            "median": np.nan,
            "std": np.nan,
            "min": np.nan,
            "max": np.nan,
            "q1": np.nan,
            "q3": np.nan,
            "iqr": np.nan,
        }

    q1 = float(np.percentile(vals, 25))
    q3 = float(np.percentile(vals, 75))
    return {
        "n": int(vals.size),
        "mean": float(np.mean(vals)),
        "median": float(np.median(vals)),
        "std": float(np.std(vals)),
        "min": float(np.min(vals)),
        "max": float(np.max(vals)),
        "q1": q1,
        "q3": q3,
        "iqr": float(q3 - q1),
    }


def add_stats(row, prefix, values):
    s = summarize(values)
    for k, v in s.items():
        row[f"{prefix}_{k}"] = v


def sample_nearest(img, yx):
    """
    Sample an image at the nearest pixel to yx = (row, col).
    """
    img = np.asarray(img)
    y = int(np.rint(yx[0]))
    x = int(np.rint(yx[1]))
    y = int(np.clip(y, 0, img.shape[0] - 1))
    x = int(np.clip(x, 0, img.shape[1] - 1))
    return float(img[y, x])


def safe_contour(ax, mask, color="red", linewidth=1.5):
    if mask is None:
        return
    if mask.shape[0] >= 2 and mask.shape[1] >= 2 and np.any(mask):
        ax.contour(mask.astype(float), levels=[0.5], colors=color, linewidths=linewidth)



def median_line(ax, values, color="black"):
    values = finite_values(values)
    if values.size:
        ax.axvline(np.median(values), color=color, linestyle=":", linewidth=2)


def rel_xy(xy, bbox):
    """
    Convert full-image (row, col) coordinates to crop-relative coordinates.
    bbox is (y0, y1, x0, x1).
    """
    if xy is None or bbox is None:
        return None
    row, col = xy
    y0, y1, x0, x1 = bbox
    return (row - y0, col - x0)


def draw_centroid_square(ax, xy, color="red", size_points=36, linewidth=1.5):
    """
    Draw a small square marker at xy = (row, col).
    The square size is in display points, so it looks similar across panels.
    """
    if xy is None:
        return

    row, col = xy
    ax.scatter(
        [col],
        [row],
        marker="s",
        s=size_points,
        facecolors="none",
        edgecolors=color,
        linewidths=linewidth,
        zorder=20,
        clip_on=False,
    )


def centroid_in_mask(mask, yx):
    """
    Check if a (row, col) coordinate falls inside a boolean mask.
    Uses nearest pixel sampling.
    """
    if mask is None or yx is None:
        return False

    y = int(np.rint(yx[0]))
    x = int(np.rint(yx[1]))

    if y < 0 or x < 0 or y >= mask.shape[0] or x >= mask.shape[1]:
        return False

    return bool(mask[y, x])


def plot_section_clean(
    fig_title,
    mask_crop,
    channel_crops,
    hist_values,
    channel_display_names,
    em_crop=None,
    em_mask_crop=None,
    lut_crop=None,
    lut_mask_crop=None,
    lut_values=None,
    summary_text="",
    norm_xlim_max=3.0,
    show_inline=False,
    centroid_red_xy=None,
    centroid_full_xy=None,
):
    """
    Layout:
      - summary + mean-normalized channels + LUT + EM

    IMPORTANT CHANGE:
      - Composite images are ONLY saved if they are also shown (show_inline=True)
      - This reduces output file size dramatically
    """

    if not SHOW_PLOTS:
        return

    n_main_cols = len(channel_display_names) + 3
    em_col = len(channel_display_names) + 2
    lut_col = len(channel_display_names) + 1

    fig = plt.figure(figsize=(4 * n_main_cols, 12))
    fig.suptitle(fig_title, y=0.995)

    gs = fig.add_gridspec(
        3,
        n_main_cols,
        left=0.04,
        right=0.97,
        top=0.94,
        bottom=0.06,
        wspace=0.25,
        hspace=0.30,
    )

    ax0 = fig.add_subplot(gs[0, 0])
    show_square_image(ax0, mask_crop, fig_title, cmap="gray")

    ax1 = fig.add_subplot(gs[1, 0])
    ax1.axis("off")
    ax1.text(0.0, 0.95, summary_text, fontsize=10, va="top")

    ax2 = fig.add_subplot(gs[2, 0])
    ax2.axis("off")

    for ch, name in enumerate(channel_display_names):
        axr = fig.add_subplot(gs[0, ch + 1])
        show_square_image(
            axr,
            channel_crops[ch],
            f"{name}\nraw bbox",
            cmap="viridis",
            norm=DISPLAY_NORM,
        )
        draw_centroid_square(axr, centroid_red_xy)
        safe_contour(axr, mask_crop)

        axm = fig.add_subplot(gs[1, ch + 1])
        masked = np.full_like(channel_crops[ch], np.nan, dtype=np.float64)
        masked[mask_crop] = channel_crops[ch][mask_crop]
        show_square_image(
            axm,
            masked,
            f"{name}\nmasked",
            cmap="viridis",
            norm=DISPLAY_NORM,
        )
        draw_centroid_square(axm, centroid_red_xy)
        safe_contour(axm, mask_crop)

        axh = fig.add_subplot(gs[2, ch + 1])
        vals = finite_values(hist_values[ch])
        if vals.size:
            axh.hist(vals, bins=30)
        axh.set_title(f"{name}\nhistogram")
        axh.set_xlabel("Normalized value")
        axh.set_ylabel("Count")
        axh.set_xlim(0, norm_xlim_max)
        median_line(axh, vals, color="black")

    if lut_crop is not None:
        ax_lut0 = fig.add_subplot(gs[0, lut_col])
        show_square_image(
            ax_lut0,
            lut_crop,
            "15N LUT raw bbox",
            cmap=CUSTOM_15N_CMAP,
            norm=NORM_15N,
        )
        draw_centroid_square(ax_lut0, centroid_red_xy)
        if lut_mask_crop is not None:
            safe_contour(ax_lut0, lut_mask_crop)

        ax_lut1 = fig.add_subplot(gs[1, lut_col])
        lut_masked = np.full_like(lut_crop, np.nan, dtype=np.float64)
        if lut_mask_crop is not None:
            lut_masked[lut_mask_crop] = lut_crop[lut_mask_crop]
        show_square_image(
            ax_lut1,
            lut_masked,
            "15N LUT masked",
            cmap=CUSTOM_15N_CMAP,
            norm=NORM_15N,
        )
        draw_centroid_square(ax_lut1, centroid_red_xy)
        if lut_mask_crop is not None:
            safe_contour(ax_lut1, lut_mask_crop)

        ax_lut2 = fig.add_subplot(gs[2, lut_col])
        vals = finite_values(lut_values if lut_values is not None else [])
        if vals.size:
            counts, bin_edges = np.histogram(vals, bins=30)
            widths = bin_edges[1:] - bin_edges[:-1]
            centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
            for c, center, width in zip(counts, centers, widths):
                ax_lut2.bar(
                    center,
                    c,
                    width=width,
                    align="center",
                    color=CUSTOM_15N_CMAP(NORM_15N(center)),
                    edgecolor="none",
                )
        ax_lut2.set_title("15N LUT\nhistogram")
        ax_lut2.set_xlabel("15N incorporation")
        ax_lut2.set_ylabel("Count")
        ax_lut2.set_xlim(GLOBAL_VMIN, GLOBAL_VMAX)
        median_line(ax_lut2, vals, color="black")

    if em_crop is not None:
        ax_em0 = fig.add_subplot(gs[0, em_col])
        show_square_image(ax_em0, em_crop, "EM raw bbox", cmap="gray")
        draw_centroid_square(ax_em0, centroid_full_xy)
        if em_mask_crop is not None:
            safe_contour(ax_em0, em_mask_crop)

        ax_em1 = fig.add_subplot(gs[1, em_col])
        em_masked = np.full_like(em_crop, np.nan, dtype=np.float64)
        if em_mask_crop is not None:
            em_masked[em_mask_crop] = em_crop[em_mask_crop]
        show_square_image(ax_em1, em_masked, "EM masked", cmap="gray")
        draw_centroid_square(ax_em1, centroid_full_xy)
        if em_mask_crop is not None:
            safe_contour(ax_em1, em_mask_crop)

        ax_em2 = fig.add_subplot(gs[2, em_col])
        ax_em2.remove()

    # Figures are displayed only (no file output)
    if show_inline and SHOW_PLOTS:
        plt.show()

    plt.close(fig)


def find_first_tiff(folder: Path):
    files = sorted(list(folder.glob("*.tif")) + list(folder.glob("*.tiff")))
    if not files:
        raise FileNotFoundError(f"No TIFF files found in {folder}")
    if len(files) > 1:
        print(f"Warning: multiple TIFF files found in {folder}, using {files[0].name}")
    return files[0]


def load_stack(stack_path: Path):
    stack = tiff.imread(str(stack_path))
    if stack.ndim != 3:
        raise ValueError(f"Expected a 3D TIFF stack, got shape {stack.shape}")

    if stack.shape[1:] == TARGET_STACK_SHAPE:
        pass
    elif stack.shape[:2] == TARGET_STACK_SHAPE:
        stack = np.moveaxis(stack, -1, 0)
    else:
        raise ValueError(f"Stack spatial shape {stack.shape[-2:]} does not match {TARGET_STACK_SHAPE}")

    if stack.shape[0] not in (6, 7):
        raise ValueError(f"Unsupported channel count after loading: {stack.shape[0]}. Expected 6 or 7.")

    return stack.astype(np.float64)


def process_sample(sample_dir: Path):
    label_dir = sample_dir / "label_images"
    stack_dir = sample_dir / "nanosims_stack"
    em_dir = sample_dir / "input_em"
    output_dir = OUTPUT_ROOT / sample_dir.name
    output_dir.mkdir(parents=True, exist_ok=True)

    label_files = sorted(list(label_dir.glob("*.tif")) + list(label_dir.glob("*.tiff")))
    if len(label_files) == 0:
        raise FileNotFoundError(f"No label TIFFs found in {label_dir}")

    stack_path = find_first_tiff(stack_dir)
    em_path = find_first_tiff(em_dir)

    print(f"\n=== Sample: {sample_dir.name} ===")
    print(f"Using stack: {stack_path.name}")
    print(f"Using EM image: {em_path.name}")
    print(f"Outputs will be saved to: {output_dir.resolve()}")

    stack = load_stack(stack_path)
    n_channels = stack.shape[0]
    print(f"Stack shape: {stack.shape}")
    print(f"Number of original channels: {n_channels}")

    em_img = tiff.imread(str(em_path))
    if em_img.ndim != 2:
        raise ValueError(f"Expected a 2D EM image, got shape {em_img.shape}")

    print(f"EM shape: {em_img.shape}")

    cell_dir = sample_dir / "label_cells"
    if not cell_dir.exists():
        raise FileNotFoundError(f"Cell label folder not found: {cell_dir}")

    cell_path = find_first_tiff(cell_dir)
    cell_img = load_cell_labels(cell_path)

    if cell_img.shape != em_img.shape:
        raise ValueError(
            f"Cell label image shape {cell_img.shape} does not match EM shape {em_img.shape} "
            f"for {cell_path.name}"
        )

    print(f"Using cell labels: {cell_path.name}")

    cell_norm_cache = {}


    raw_channel_names = infer_channel_names(n_channels)
    channel_index = {name: i for i, name in enumerate(raw_channel_names)}

    if "12C14N" not in channel_index or "12C15N" not in channel_index:
        raise ValueError("Required channels 12C14N and 12C15N were not found in the TIFF stack.")

    ch2_idx = channel_index["12C14N"]
    ch3_idx = channel_index["12C15N"]

    stack_mean_norm, field_means = whole_field_mean_norm(stack, raw_channel_names)
    mean_norm_channel_names = [f"{name}_mean_norm" for name in raw_channel_names]

    print("\nWhole-field mean normalization factors:")
    for name, mu in zip(raw_channel_names, field_means):
        print(f"{name}: {mu:.6f}")

    summed = stack[ch2_idx] + stack[ch3_idx]
    fractional_15N = np.full_like(summed, np.nan, dtype=np.float64)
    np.divide(stack[ch3_idx], summed, out=fractional_15N, where=(summed != 0))

    sum_finite = summed[np.isfinite(summed)]
    sum_mean = float(np.mean(sum_finite)) if sum_finite.size else 1.0
    if not np.isfinite(sum_mean) or sum_mean == 0:
        print(f"Warning: summed 12C14N+12C15N channel has invalid mean {sum_mean}. Using 1.0.")
        sum_mean = 1.0

    summed_mean_norm = summed / sum_mean
    summed_mean_norm = np.asarray(summed_mean_norm, dtype=np.float64)
    sum_channel_name = "sum_12C14N_12C15N_mean_norm"

    plot_stack = np.concatenate([stack_mean_norm, summed_mean_norm[None]], axis=0)
    plot_channel_names = mean_norm_channel_names + [sum_channel_name]

    qc_row = {
        "sample": sample_dir.name,
        "stack": stack_path.name,
        "em": em_path.name,
        "fractional_15N_pixels": int(np.isfinite(fractional_15N).sum()),
        "sum_12C14N_12C15N_mean": sum_mean,
    }
    for k, v in summarize(fractional_15N).items():
        qc_row[f"fractional_15N_{k}"] = v

    for i, name in enumerate(raw_channel_names):
        raw_s = summarize(stack[i])
        mean_s = summarize(stack_mean_norm[i])
        for k, v in raw_s.items():
            qc_row[f"whole_field_raw_{name}_{k}"] = v
        for k, v in mean_s.items():
            qc_row[f"whole_field_mean_norm_{name}_{k}"] = v

    sum_mean_s = summarize(summed_mean_norm)
    for k, v in sum_mean_s.items():
        qc_row[f"sum_12C14N_12C15N_mean_norm_{k}"] = v

    print("\nQC Whole-field 15N:")
    print(f"Pixels: {qc_row['fractional_15N_pixels']}")
    print(f"Mean:   {qc_row['fractional_15N_mean']:.6f}")
    print(f"Median: {qc_row['fractional_15N_median']:.6f}")
    print(f"Std:    {qc_row['fractional_15N_std']:.6f}")

    display(pd.DataFrame([qc_row]))

    all_rows = []

    for label_path in tqdm(label_files, desc=f"{sample_dir.name} label images"):
        print(f"\nProcessing {label_path.name}")
        labels = tiff.imread(str(label_path))
        if labels.ndim != 2:
            raise ValueError(f"Expected 2D label TIFF, got shape {labels.shape} for {label_path.name}")

        label_vals = np.unique(labels)
        label_vals = label_vals[(label_vals != 0) & np.isfinite(label_vals)]
        label_vals = np.rint(label_vals).astype(int)
        print(f"Labels in {label_path.name}: {label_vals}")

        scale_y = round(labels.shape[0] / TARGET_STACK_SHAPE[0])
        scale_x = round(labels.shape[1] / TARGET_STACK_SHAPE[1])
        if scale_y != scale_x:
            raise ValueError(f"Inconsistent inferred scale for {label_path.name}: {scale_y} vs {scale_x}")
        scale = int(scale_y)
        print(f"Inferred full-res scale: {scale}x")

        group_rows = []


        for label_idx, lab in enumerate(tqdm(label_vals, desc=f"{label_path.stem}", leave=False)):
            lab = int(lab)
            original_mask = labels == lab
            if not np.any(original_mask):
                continue

            strict_reduced_mask = strict_reduce_mask_to_shape(original_mask, TARGET_STACK_SHAPE)
            if not np.any(strict_reduced_mask):
                continue

            footprint = np.ones((2 * ERODE_PIXELS + 1, 2 * ERODE_PIXELS + 1), dtype=bool)
            eroded_mask = binary_erosion(strict_reduced_mask, footprint=footprint)

            strict_bbox = bbox_of_mask(strict_reduced_mask)
            if strict_bbox is None:
                continue
            sy0, sy1, sx0, sx1 = strict_bbox
            strict_mask_crop = strict_reduced_mask[sy0:sy1, sx0:sx1]

            original_bbox = bbox_of_mask(original_mask)
            if original_bbox is None:
                continue
            oy0, oy1, ox0, ox1 = original_bbox
            em_crop = em_img[oy0:oy1, ox0:ox1]

            strict_fullres_mask = expand_reduced_mask_to_fullres(strict_reduced_mask, labels.shape)
            strict_fullres_mask_crop = strict_fullres_mask[oy0:oy1, ox0:ox1]

            props = regionprops(original_mask.astype(np.uint8))
            if len(props) == 0:
                continue
            prop = props[0]

            major_px = float(prop.major_axis_length)
            minor_px = float(prop.minor_axis_length)

            centroid_y_full = float(prop.centroid[0])
            centroid_x_full = float(prop.centroid[1])
            centroid_y_red = centroid_y_full / scale
            centroid_x_red = centroid_x_full / scale

            centroid_red_xy = (centroid_y_red, centroid_x_red)
            centroid_full_xy = (centroid_y_full, centroid_x_full)

            centroid_in_strict_mask = centroid_in_mask(strict_reduced_mask, centroid_red_xy)

            # Cell assignment from centroid overlap
            cell_label_id = int(sample_nearest(cell_img, centroid_full_xy))
            if cell_label_id > 0:
                cell_region_mask_full = (cell_img == cell_label_id)
                cell_region_pixel_count = int(np.count_nonzero(cell_region_mask_full))
                centroid_in_cell_mask = centroid_in_mask(cell_region_mask_full, centroid_full_xy)

                # Downsample the full-res cell mask to stack resolution for normalization
                # so it matches the 256 x 256 NanoSIMS images.
                cell_region_mask = strict_reduce_mask_to_shape(
                    cell_region_mask_full, TARGET_STACK_SHAPE
                )
                if not np.any(cell_region_mask):
                    cell_region_mask = None
            else:
                cell_region_mask_full = None
                cell_region_mask = None
                cell_region_pixel_count = 0
                centroid_in_cell_mask = False

            # Cell-based normalization cache
            if cell_label_id > 0 and cell_region_mask is not None and np.any(cell_region_mask):
                if cell_label_id not in cell_norm_cache:
                    cell_stack_mean_norm, cell_field_means = whole_field_mean_norm_masked(
                        stack, raw_channel_names, cell_region_mask
                    )

                    cell_summed = stack[ch2_idx] + stack[ch3_idx]
                    cell_sum_vals = cell_summed[cell_region_mask]
                    cell_sum_mean = float(np.mean(cell_sum_vals)) if cell_sum_vals.size else 1.0
                    if not np.isfinite(cell_sum_mean) or cell_sum_mean == 0:
                        print(
                            f"Warning: cell summed 12C14N+12C15N channel has invalid mean {cell_sum_mean}. Using 1.0."
                        )
                        cell_sum_mean = 1.0

                    cell_summed_mean_norm = cell_summed / cell_sum_mean

                    cell_norm_cache[cell_label_id] = (
                        cell_stack_mean_norm,
                        cell_field_means,
                        cell_summed_mean_norm,
                        cell_sum_mean,
                    )

                cell_stack_mean_norm, cell_field_means, cell_summed_mean_norm, cell_sum_mean = cell_norm_cache[cell_label_id]
            else:
                cell_stack_mean_norm = None
                cell_field_means = None
                cell_summed_mean_norm = None
                cell_sum_mean = np.nan


            strict_centroid_red_local = rel_xy(centroid_red_xy, strict_bbox)
            strict_centroid_full_local = rel_xy(centroid_full_xy, original_bbox)

            # --- stats (field-based) ---
            strict_raw_vals_per_channel = [stack[ch][strict_reduced_mask] for ch in range(n_channels)]
            strict_mean_norm_vals_per_channel = [stack_mean_norm[ch][strict_reduced_mask] for ch in range(n_channels)]
            strict_sum_vals = summed[strict_reduced_mask]
            strict_sum_mean_norm_vals = summed_mean_norm[strict_reduced_mask]
            strict_frac_vals = fractional_15N[strict_reduced_mask]

            eroded_raw_vals_per_channel = (
                [stack[ch][eroded_mask] for ch in range(n_channels)]
                if np.any(eroded_mask)
                else [np.array([]) for _ in range(n_channels)]
            )
            eroded_mean_norm_vals_per_channel = (
                [stack_mean_norm[ch][eroded_mask] for ch in range(n_channels)]
                if np.any(eroded_mask)
                else [np.array([]) for _ in range(n_channels)]
            )
            eroded_sum_vals = summed[eroded_mask] if np.any(eroded_mask) else np.array([])
            eroded_sum_mean_norm_vals = summed_mean_norm[eroded_mask] if np.any(eroded_mask) else np.array([])
            eroded_frac_vals = fractional_15N[eroded_mask] if np.any(eroded_mask) else np.array([])

            # --- stats (cell-based) ---
            if cell_stack_mean_norm is not None:
                strict_cell_norm_vals_per_channel = [
                    cell_stack_mean_norm[ch][strict_reduced_mask] for ch in range(n_channels)
                ]
                strict_cell_sum_mean_norm_vals = cell_summed_mean_norm[strict_reduced_mask]

                eroded_cell_norm_vals_per_channel = (
                    [cell_stack_mean_norm[ch][eroded_mask] for ch in range(n_channels)]
                    if np.any(eroded_mask)
                    else [np.array([]) for _ in range(n_channels)]
                )
                eroded_cell_sum_mean_norm_vals = (
                    cell_summed_mean_norm[eroded_mask] if np.any(eroded_mask) else np.array([])
                )
            else:
                strict_cell_norm_vals_per_channel = [np.array([]) for _ in range(n_channels)]
                strict_cell_sum_mean_norm_vals = np.array([])
                eroded_cell_norm_vals_per_channel = [np.array([]) for _ in range(n_channels)]
                eroded_cell_sum_mean_norm_vals = np.array([])

            # --- row metadata ---
            row = {
                "sample": sample_dir.name,
                "label_id": int(lab),
                "source_label_file": label_path.name,
                "source_stack_file": stack_path.name,
                "source_em_file": em_path.name,
                "source_cell_file": cell_path.name,
                "cell_label_id": int(cell_label_id),
                "cell_region_pixel_count": cell_region_pixel_count,
                "centroid_in_cell_mask": centroid_in_cell_mask,
                "pixel_size_nm": PIXEL_SIZE_NM,
                "normalization_method": "whole_field_mean",
                "cell_normalization_method": "cell_mean" if cell_label_id > 0 else "none",
                "original_label_pixel_count": int(np.count_nonzero(original_mask)),
                "strict_reduced_pixel_count": int(np.count_nonzero(strict_reduced_mask)),
                "strict_eroded_pixel_count": int(np.count_nonzero(eroded_mask)),
                "area_nm2": float(prop.area * PIXEL_SIZE_NM**2),
                "perimeter_nm": float(prop.perimeter * PIXEL_SIZE_NM),
                "aspect_ratio": float(major_px / minor_px) if minor_px > 0 else np.nan,
                "major_axis_length_px": major_px,
                "minor_axis_length_px": minor_px,
                "major_axis_length_nm": float(major_px * PIXEL_SIZE_NM),
                "minor_axis_length_nm": float(minor_px * PIXEL_SIZE_NM),
                "centroid_x_px_fullres": centroid_x_full,
                "centroid_y_px_fullres": centroid_y_full,
                "centroid_x_px_reduced": centroid_x_red,
                "centroid_y_px_reduced": centroid_y_red,
                "centroid_in_strict_reduced_mask": centroid_in_strict_mask,
                "cell_12C14N_12C15N_mean": cell_sum_mean,
            }

            # --- stats writing (field-based) ---
            for ch, name in enumerate(raw_channel_names):
                add_stats(row, f"strict_reduced_raw_{name}", strict_raw_vals_per_channel[ch])
                add_stats(row, f"strict_reduced_mean_norm_{name}", strict_mean_norm_vals_per_channel[ch])
                add_stats(row, f"strict_eroded_raw_{name}", eroded_raw_vals_per_channel[ch])
                add_stats(row, f"strict_eroded_mean_norm_{name}", eroded_mean_norm_vals_per_channel[ch])

            add_stats(row, "strict_reduced_raw_sum_12C14N_12C15N", strict_sum_vals)
            add_stats(row, "strict_reduced_mean_norm_sum_12C14N_12C15N", strict_sum_mean_norm_vals)
            add_stats(row, "strict_reduced_raw_fractional_15N", strict_frac_vals)

            add_stats(row, "strict_eroded_raw_sum_12C14N_12C15N", eroded_sum_vals)
            add_stats(row, "strict_eroded_mean_norm_sum_12C14N_12C15N", eroded_sum_mean_norm_vals)
            add_stats(row, "strict_eroded_raw_fractional_15N", eroded_frac_vals)

            # --- stats writing (cell-based) ---
            for ch, name in enumerate(raw_channel_names):
                add_stats(row, f"strict_reduced_cell_norm_{name}", strict_cell_norm_vals_per_channel[ch])
                add_stats(row, f"strict_eroded_cell_norm_{name}", eroded_cell_norm_vals_per_channel[ch])

                row[f"centroid_strict_reduced_cell_norm_{name}"] = (
                    sample_nearest(cell_stack_mean_norm[ch], centroid_full_xy)
                    if cell_stack_mean_norm is not None
                    else np.nan
                )
                row[f"centroid_strict_eroded_cell_norm_{name}"] = (
                    sample_nearest(cell_stack_mean_norm[ch], centroid_full_xy)
                    if cell_stack_mean_norm is not None
                    else np.nan
                )

            add_stats(row, "strict_reduced_cell_norm_sum_12C14N_12C15N", strict_cell_sum_mean_norm_vals)
            add_stats(row, "strict_eroded_cell_norm_sum_12C14N_12C15N", eroded_cell_sum_mean_norm_vals)

            row["strict_reduced_cell_norm_sum_12C14N_12C15N_values"] = strict_cell_sum_mean_norm_vals.tolist()
            row["strict_eroded_cell_norm_sum_12C14N_12C15N_values"] = eroded_cell_sum_mean_norm_vals.tolist()

            row["centroid_strict_reduced_cell_norm_sum_12C14N_12C15N"] = (
                sample_nearest(cell_summed_mean_norm, centroid_full_xy)
                if cell_summed_mean_norm is not None
                else np.nan
            )
            row["centroid_strict_eroded_cell_norm_sum_12C14N_12C15N"] = (
                sample_nearest(cell_summed_mean_norm, centroid_full_xy)
                if cell_summed_mean_norm is not None
                else np.nan
            )

            npz_path = save_label_data_npz(
                npz_root=NPZ_ROOT,
                label_stem=label_path.stem,
                lab=lab,
                sample=sample_dir.name,
                source_label_file=label_path.name,
                strict_bbox=(sy0, sy1, sx0, sx1),
                original_bbox=(oy0, oy1, ox0, ox1),
                stack=stack,
                stack_mean_norm=stack_mean_norm,
                summed=summed,
                fractional_15N=fractional_15N,
                em_img=em_img,
                strict_mask_crop=strict_mask_crop,
                channel_names=raw_channel_names,
                field_means=field_means,
                cell_label_id=cell_label_id,
                source_cell_file=cell_path.name,
            )

            row["npz_file"] = npz_path.name

            # Add the completed row to both accumulators
            group_rows.append(row.copy())
            all_rows.append(row.copy())


            # =============================
            # plotting (unchanged)
            # =============================
            strict_channel_crops = [plot_stack[ch][sy0:sy1, sx0:sx1] for ch in range(len(plot_channel_names))]
            strict_hist_values = strict_mean_norm_vals_per_channel + [strict_sum_mean_norm_vals]

            strict_summary = (
                f"Label {lab}\n"
                f"Strict reduced pixels: {row['strict_reduced_pixel_count']}\n"
                f"Strict 15N mean: {summarize(strict_frac_vals)['mean']:.6f}\n"
                f"Strict 15N median: {summarize(strict_frac_vals)['median']:.6f}\n"
                f"Strict 15N stdev: {summarize(strict_frac_vals)['std']:.6f}"
            )

            out_prefix = None

            plot_section_clean(
                fig_title=f"{label_path.name} — Label {lab} — Strict reduced mask",
                mask_crop=strict_mask_crop,
                channel_crops=strict_channel_crops,
                hist_values=strict_hist_values,
                channel_display_names=plot_channel_names,
                em_crop=em_crop,
                em_mask_crop=strict_fullres_mask_crop,
                lut_crop=crop_to_bbox(fractional_15N, strict_bbox),
                lut_mask_crop=strict_mask_crop,
                lut_values=strict_frac_vals,
                summary_text=strict_summary,
                norm_xlim_max=3.0,
                show_inline=(label_idx < 4),
                centroid_red_xy=strict_centroid_red_local,
                centroid_full_xy=strict_centroid_full_local,
            )


            if np.any(eroded_mask):
                eroded_summary = (
                    f"Label {lab}\n"
                    f"Eroded reduced pixels: {row['strict_eroded_pixel_count']}\n"
                    f"Eroded 15N mean: {summarize(eroded_frac_vals)['mean']:.6f}\n"
                    f"Eroded 15N median: {summarize(eroded_frac_vals)['median']:.6f}\n"
                    f"Eroded 15N stdev: {summarize(eroded_frac_vals)['std']:.6f}"
                )
                if label_idx < 2:
                    eroded_bbox = bbox_of_mask(eroded_mask)
                    if eroded_bbox is not None:
                        ey0, ey1, ex0, ex1 = eroded_bbox
                        eroded_mask_crop = eroded_mask[ey0:ey1, ex0:ex1]
                        eroded_channel_crops = [
                            plot_stack[ch][ey0:ey1, ex0:ex1] for ch in range(len(plot_channel_names))
                        ]

                        eroded_centroid_red_local = rel_xy(centroid_red_xy, eroded_bbox)
                        eroded_centroid_full_local = rel_xy(centroid_full_xy, original_bbox)

                        plot_section_clean(
                            fig_title=f"{label_path.name} — Label {lab} — Eroded reduced mask (QC only)",
                            mask_crop=eroded_mask_crop,
                            channel_crops=eroded_channel_crops,
                            hist_values=eroded_mean_norm_vals_per_channel + [eroded_sum_mean_norm_vals],
                            channel_display_names=plot_channel_names,
                            em_crop=em_crop,
                            em_mask_crop=None,
                            lut_crop=crop_to_bbox(fractional_15N, eroded_bbox),
                            lut_mask_crop=eroded_mask_crop,
                            lut_values=eroded_frac_vals,
                            summary_text=eroded_summary,
                            norm_xlim_max=3.0,
                            show_inline=True,
                            centroid_red_xy=eroded_centroid_red_local,
                            centroid_full_xy=eroded_centroid_full_local,
                        )

        if group_rows:
            group_df = (
                pd.DataFrame(group_rows)
                .sort_values(["sample", "source_label_file", "label_id"])
                .reset_index(drop=True)
            )
        else:
            group_df = pd.DataFrame(
                columns=[
                    "sample",
                    "label_id",
                    "source_label_file",
                    "source_stack_file",
                    "source_em_file",
                ]
            )

        group_csv_path = output_dir / f"{label_path.stem}_measurements.csv"
        group_df.to_csv(group_csv_path, index=False)
        print(f"Saved group dataframe to: {group_csv_path.resolve()}")

    sample_df = pd.DataFrame(all_rows)
    if not sample_df.empty:
        sample_df = sample_df.sort_values(["sample", "source_label_file", "label_id"]).reset_index(drop=True)

    sample_csv_path = output_dir / f"{sample_dir.name}_label_measurements_mean_norm_strict_names.csv"
    sample_df.to_csv(sample_csv_path, index=False)
    print(f"Saved sample dataframe to: {sample_csv_path.resolve()}")

    return qc_row, sample_df


def main():
    sample_dirs = sorted([p for p in INPUT_ROOT.iterdir() if p.is_dir()])
    if not sample_dirs:
        raise FileNotFoundError(f"No sample subfolders found in {INPUT_ROOT.resolve()}")

    all_samples = []
    qc_rows = []

    for sample_dir in sample_dirs:
        label_dir = sample_dir / "label_images"
        stack_dir = sample_dir / "nanosims_stack"
        em_dir = sample_dir / "input_em"

        if not label_dir.exists() or not stack_dir.exists() or not em_dir.exists():
            print(f"Skipping {sample_dir.name}: missing one of input_em / label_images / nanosims_stack")
            continue

        qc_row, sample_df = process_sample(sample_dir)
        qc_rows.append(qc_row)

        if sample_df is not None and not sample_df.empty:
            all_samples.append(sample_df)

    if len(all_samples) == 0:
        print("No valid rows were produced across samples.")
        return pd.DataFrame()

    final_df = pd.concat(all_samples, ignore_index=True)
    final_df = final_df.sort_values(["sample", "source_label_file", "label_id"]).reset_index(drop=True)

    display(
        final_df[
            [
                "sample",
                "label_id",
                "source_label_file",
                "source_stack_file",
                "source_em_file",
                "pixel_size_nm",
                "normalization_method",
                "original_label_pixel_count",
                "strict_reduced_pixel_count",
                "strict_eroded_pixel_count",
                "area_nm2",
                "perimeter_nm",
                "aspect_ratio",
                "major_axis_length_px",
                "minor_axis_length_px",
                "major_axis_length_nm",
                "minor_axis_length_nm",
                "centroid_x_px_fullres",
                "centroid_y_px_fullres",
                "centroid_x_px_reduced",
                "centroid_y_px_reduced",
                "strict_reduced_raw_fractional_15N_mean",
                "strict_reduced_raw_fractional_15N_median",
                "strict_reduced_raw_fractional_15N_std",
                "strict_eroded_raw_fractional_15N_mean",
                "strict_eroded_raw_fractional_15N_median",
                "strict_eroded_raw_fractional_15N_std",
                "source_cell_file",
                "cell_label_id",
                "cell_region_pixel_count",
                "centroid_in_cell_mask",
                "cell_12C14N_12C15N_mean",
                "cell_normalization_method",
                "strict_reduced_cell_norm_sum_12C14N_12C15N_mean",
                "strict_eroded_cell_norm_sum_12C14N_12C15N_mean",
            ]
        ]
    )

    csv_path = OUTPUT_ROOT / "all_samples_label_measurements_mean_norm_strict_names.csv"
    final_df.to_csv(csv_path, index=False)
    print(f"\nSaved combined dataframe to: {csv_path.resolve()}")

    qc_df = pd.DataFrame(qc_rows)
    qc_path = OUTPUT_ROOT / "all_samples_qc.csv"
    qc_df.to_csv(qc_path, index=False)
    print(f"Saved combined QC summary to: {qc_path.resolve()}")

    return final_df


if __name__ == "__main__":
    df = main()

C:\Users\narendradp\AppData\Local\Temp\ipykernel_50916\3925331938.py:100: UserWarning: Overwriting the cmap 'fiji_lut_from_csv' that was already in the registry.
  mpl.colormaps.register(cmap, name="fiji_lut_from_csv", force=True)


Loaded LUT from LUT.csv (256 entries).
Skipping old: missing one of input_em / label_images / nanosims_stack

=== Sample: S18_Ar2 ===
Using stack: 030625-DNp3e2c8-S18-ROI2.nrrd.tif
Using EM image: S18 Ar2 EM aligned.tif
Outputs will be saved to: C:\Users\narendradp\OneDrive - National Institutes of Health\Documents\Jennifer_Data\2D_measure_NanoSIMS_using_labels\output\S18_Ar2
Stack shape: (7, 256, 256)
Number of original channels: 7
EM shape: (8567, 8567)
Detected binary cell mask in Cell_S18_Ar2.tif; converting connected components to labels.


C:\Users\narendradp\AppData\Local\Temp\ipykernel_50916\3925331938.py:241: FutureWarning: Parameter `area_threshold` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_holes`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  mask = remove_small_holes(mask, area_threshold=mask.size)


Using cell labels: Cell_S18_Ar2.tif

Whole-field mean normalization factors:
16O: 417.178131
12C21H: 789.539444
12C14N: 3352.454788
12C15N: 896.260498
29Si: 0.004486
31P: 0.870438
32S: 40.208115

QC Whole-field 15N:
Pixels: 65536
Mean:   0.209932
Median: 0.215821
Std:    0.031936


,sample,stack,em,fractional_15N_pixels,sum_12C14N_12C15N_mean,fractional_15N_n,fractional_15N_mean,fractional_15N_median,fractional_15N_std,fractional_15N_min,...,whole_field_mean_norm_32S_iqr,sum_12C14N_12C15N_mean_norm_n,sum_12C14N_12C15N_mean_norm_mean,sum_12C14N_12C15N_mean_norm_median,sum_12C14N_12C15N_mean_norm_std,sum_12C14N_12C15N_mean_norm_min,sum_12C14N_12C15N_mean_norm_max,sum_12C14N_12C15N_mean_norm_q1,sum_12C14N_12C15N_mean_norm_q3,sum_12C14N_12C15N_mean_norm_iqr
0,S18_Ar2,030625-DNp3e2c8-S18-ROI2.nrrd.tif,S18 Ar2 EM aligned.tif,65536,4248.715286,65536,0.209932,0.215821,0.031936,0.012838,...,0.273577,65536,1.0,1.023839,0.195729,0.014122,3.428566,0.936989,1.093978,0.156989


...

PL-Speckled_S2_Ar5:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\narendradp\AppData\Local\Temp\ipykernel_50916\3925331938.py:851: FutureWarning: `binary_erosion` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.erosion` instead. Note the pixel shift by 1 for even-sized footprints (see docstring notes).
  eroded_mask = binary_erosion(strict_reduced_mask, footprint=footprint)
C:\Users\narendradp\AppData\Local\Temp\ipykernel_50916\3925331938.py:873: FutureWarning: `RegionProperties.major_axis_length` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.axis_major_length` instead. 
  major_px = float(prop.major_axis_length)
C:\Users\narendradp\AppData\Local\Temp\ipykernel_50916\3925331938.py:874: FutureWarning: `RegionProperties.minor_axis_length` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.axis_minor_length` instead. 
  minor_px = float(prop.minor_axis_length)


Saved group dataframe to: C:\Users\narendradp\OneDrive - National Institutes of Health\Documents\Jennifer_Data\2D_measure_NanoSIMS_using_labels\output\S2_Ar5\PL-Speckled_S2_Ar5_measurements.csv
Saved sample dataframe to: C:\Users\narendradp\OneDrive - National Institutes of Health\Documents\Jennifer_Data\2D_measure_NanoSIMS_using_labels\output\S2_Ar5\S2_Ar5_label_measurements_mean_norm_strict_names.csv


,sample,label_id,source_label_file,source_stack_file,source_em_file,pixel_size_nm,normalization_method,original_label_pixel_count,strict_reduced_pixel_count,strict_eroded_pixel_count,...,strict_eroded_raw_fractional_15N_median,strict_eroded_raw_fractional_15N_std,source_cell_file,cell_label_id,cell_region_pixel_count,centroid_in_cell_mask,cell_12C14N_12C15N_mean,cell_normalization_method,strict_reduced_cell_norm_sum_12C14N_12C15N_mean,strict_eroded_cell_norm_sum_12C14N_12C15N_mean
0,S18_Ar2,10001,Mitochondria_S18_Ar2.tif,030625-DNp3e2c8-S18-ROI2.nrrd.tif,S18 Ar2 EM aligned.tif,1.751,whole_field_mean,31543,15,0,...,NaN,NaN,Cell_S18_Ar2.tif,1,73393489,True,4248.715286,cell_mean,1.092142,NaN
1,S18_Ar2,10003,Mitochondria_S18_Ar2.tif,030625-DNp3e2c8-S18-ROI2.nrrd.tif,S18 Ar2 EM aligned.tif,1.751,whole_field_mean,356559,257,132,...,0.232013,0.009143,Cell_S18_Ar2.tif,1,73393489,True,4248.715286,cell_mean,1.099715,1.104887
2,S18_Ar2,10005,Mitochondria_S18_Ar2.tif,030625-DNp3e2c8-S18-ROI2.nrrd.tif,S18 Ar2 EM aligned.tif,1.751,whole_field_mean,23263,12,0,...,NaN,NaN,Cell_S18_Ar2.tif,1,73393489,True,4248.715286,cell_mean,1.017916,NaN
3,S18_Ar2,10015,Mitochondria_S18_Ar2.tif,030625-DNp3e2c8-S18-ROI2.nrrd.tif,S18 Ar2 EM aligned.tif,1.751,whole_field_mean,34438,21,3,...,0.219369,0.006046,Cell_S18_Ar2.tif,1,73393489,True,4248.715286,cell_mean,1.072873,1.093193
4,S18_Ar2,10021,Mitochondria_S18_Ar2.tif,030625-DNp3e2c8-S18-ROI2.nrrd.tif,S18 Ar2 EM aligned.tif,1.751,whole_field_mean,239454,162,71,...,0.227756,0.009144,Cell_S18_Ar2.tif,1,73393489,True,4248.715286,cell_mean,1.156479,1.171741
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6092,S2_Ar5,3,Mitolysosome_S2_Ar5.tif,043024-DN-p3e1c5-sample2-Ar5.nrrd.tif,Mitolysosome1-Contents_S2_Ar5.tif,1.751,whole_field_mean,487602,619,498,...,0.337275,0.067242,Cell_S2_Ar5.tif,1,46977316,True,27244.560120,cell_mean,1.005277,0.990401
6093,S2_Ar5,1,Mitophagophore1-Contents_S2_Ar5.tif,043024-DN-p3e1c5-sample2-Ar5.nrrd.tif,Mitolysosome1-Contents_S2_Ar5.tif,1.751,whole_field_mean,32715,32,10,...,0.478745,0.001254,Cell_S2_Ar5.tif,1,46977316,True,27244.560120,cell_mean,1.019948,1.009192
6094,S2_Ar5,1,Mitophagophore_S2_Ar5.tif,043024-DN-p3e1c5-sample2-Ar5.nrrd.tif,Mitolysosome1-Contents_S2_Ar5.tif,1.751,whole_field_mean,39627,42,16,...,0.478646,0.001127,Cell_S2_Ar5.tif,1,46977316,True,27244.560120,cell_mean,1.024276,1.014040
6095,S2_Ar5,1,Mitoplast_S2_Ar5.tif,043024-DN-p3e1c5-sample2-Ar5.nrrd.tif,Mitolysosome1-Contents_S2_Ar5.tif,1.751,whole_field_mean,26307,25,6,...,0.468198,0.003546,Cell_S2_Ar5.tif,1,46977316,True,27244.560120,cell_mean,1.000322,0.978587



Saved combined dataframe to: C:\Users\narendradp\OneDrive - National Institutes of Health\Documents\Jennifer_Data\2D_measure_NanoSIMS_using_labels\output\all_samples_label_measurements_mean_norm_strict_names.csv
Saved combined QC summary to: C:\Users\narendradp\OneDrive - National Institutes of Health\Documents\Jennifer_Data\2D_measure_NanoSIMS_using_labels\output\all_samples_qc.csv


In [1]:
# =============================
# Combined annotation script
#   - keeps the SAME Mitolysosome<->contents mapping logic
#   - adds unique_ID using label_id
#   - annotates partially wrapped rows
#   - adds group names from key.csv
#   - writes combined_output.csv next to the script
# =============================

from pathlib import Path
import numpy as np
import pandas as pd

# -----------------------------
# Paths
# -----------------------------
SCRIPT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
INPUT_PATH = SCRIPT_DIR / "output" / "all_samples_label_measurements_mean_norm_strict_names.csv"
KEY_PATH = SCRIPT_DIR / "key.csv"
PW_PATH = SCRIPT_DIR / "Partially_Wrapped_Masterlist.csv"
OUT_PATH = SCRIPT_DIR / "combined_output_before_annotation.csv"

# -----------------------------
# Load main dataframe
# -----------------------------
df = pd.read_csv(INPUT_PATH).copy()
df["__row_order__"] = np.arange(len(df))

if "sample" not in df.columns:
    raise ValueError("Expected a 'sample' column in the main input dataframe.")
if "source_label_file" not in df.columns:
    raise ValueError("Expected a 'source_label_file' column in the main input dataframe.")
if "label_id" not in df.columns:
    raise ValueError("Expected a 'label_id' column in the main input dataframe.")

df["sample"] = df["sample"].astype(str)
df["source_label_file"] = df["source_label_file"].astype(str)
df["label_id"] = pd.to_numeric(df["label_id"], errors="coerce").astype("Int64")

# -----------------------------
# Add group names from key.csv
# -----------------------------
key_df = pd.read_csv(KEY_PATH).copy()
key_df.columns = [c.strip() for c in key_df.columns]

if "sample" not in key_df.columns:
    raise ValueError("key.csv must contain a 'sample' column.")

key_df["sample"] = key_df["sample"].astype(str)
df = df.merge(key_df, on="sample", how="left", validate="m:1")

if "experiment_name" in df.columns:
    df["group_name"] = df["experiment_name"]
elif "sample_name" in df.columns:
    df["group_name"] = df["sample_name"]
else:
    df["group_name"] = df["sample"]

# -----------------------------
# unique_ID
# -----------------------------
base_name = df["source_label_file"].str.replace(r"\.tif$", "", regex=True)
df["unique_ID"] = base_name + "_" + df["label_id"].astype(str)

# -----------------------------
# Partially wrapped annotation
# -----------------------------
pw = pd.read_csv(PW_PATH).copy()
pw.columns = [c.strip() for c in pw.columns]

if "unique_ID" not in pw.columns:
    raise ValueError("Partially_Wrapped_Masterlist.csv must contain a 'unique_ID' column.")

pw["unique_ID"] = pw["unique_ID"].astype(str)

if "mask" in pw.columns:
    pw["partially_wrapped_group_name"] = (
        pw["mask"].astype(str)
        .str.replace(r"\d+", "", regex=True)
        .str.replace("-Contents", "", regex=False)
    )
else:
    pw["partially_wrapped_group_name"] = pd.NA

pw_keep_cols = ["unique_ID"]
for col in ["mask", "source_label_file", "label", "napari label", "partially_wrapped_group_name"]:
    if col in pw.columns:
        pw_keep_cols.append(col)

pw_annot = pw[pw_keep_cols].rename(columns={
    "mask": "pw_mask",
    "source_label_file": "pw_source_label_file",
    "label": "pw_label",
    "napari label": "pw_napari_label",
})

df = df.merge(pw_annot, on="unique_ID", how="left", validate="m:1")
df["partially_wrapped"] = df["pw_source_label_file"].notna()

# -----------------------------
# SAME Mitolysosome mapping logic as the prior composite script
# -----------------------------
src = df["source_label_file"].astype(str)

df["mito_kind"] = np.select(
    [
        src.str.match(r"^Mitolysosome\d+-Contents_.+\.tif$", na=False),
        src.str.match(r"^Mitolysosome_.+\.tif$", na=False),
        src.str.match(r"^Mitolysosome\d+-(?!Contents).+\.tif$", na=False),
    ],
    [
        "contents",
        "main",
        "other_mito_related",
    ],
    default="other",
)

# Create corrected_mito_rank column in df first
df["corrected_mito_rank"] = pd.Series(pd.NA, index=df.index, dtype="Int64")

# Main rows: sequential order within each sample, based on row order
main = df[df["mito_kind"] == "main"].copy()
main = main.sort_values(["sample", "__row_order__"])
main["corrected_mito_rank"] = main.groupby("sample").cumcount() + 1
df.loc[main.index, "corrected_mito_rank"] = main["corrected_mito_rank"].astype("Int64")

# Contents rows: rank comes from file name
contents = df[df["mito_kind"] == "contents"].copy()
contents_rank = (
    contents["source_label_file"]
    .str.extract(r"^Mitolysosome(?P<n>\d+)-Contents_.+\.tif$")["n"]
    .astype("Int64")
)
df.loc[contents.index, "corrected_mito_rank"] = contents_rank

# Parent lookup from main rows
main_lookup = main[[
    "sample",
    "corrected_mito_rank",
    "label_id",
    "source_label_file",
    "unique_ID",
    "group_name",
]].copy()

main_lookup = main_lookup.rename(columns={
    "label_id": "corrected_mito_label_id",
    "source_label_file": "corrected_mito_source_label_file",
    "unique_ID": "corrected_mito_unique_ID",
    "group_name": "corrected_mito_group_name",
})

main_lookup["corrected_mito_unique_ID"] = (
    main_lookup["corrected_mito_source_label_file"].astype(str).str.replace(r"\.tif$", "", regex=True)
    + "_"
    + main_lookup["corrected_mito_label_id"].astype(str)
)

# Merge parent annotations onto every row using sample + corrected_mito_rank
df = df.merge(
    main_lookup,
    on=["sample", "corrected_mito_rank"],
    how="left",
    validate="m:1"
)

# Parent rows map to themselves
main_rows = df["mito_kind"].eq("main")
df.loc[main_rows, "corrected_mito_label_id"] = df.loc[main_rows, "label_id"]
df.loc[main_rows, "corrected_mito_source_label_file"] = df.loc[main_rows, "source_label_file"]
df.loc[main_rows, "corrected_mito_unique_ID"] = df.loc[main_rows, "unique_ID"]
df.loc[main_rows, "corrected_mito_group_name"] = df.loc[main_rows, "group_name"]

# Convenience key
df["corrected_mito_key"] = np.where(
    df["corrected_mito_rank"].notna(),
    df["sample"].astype(str) + "_M" + df["corrected_mito_rank"].astype("Int64").astype(str),
    pd.NA
)

# Flags
df["is_main_mitolysosome"] = df["mito_kind"].eq("main")
df["is_mitolysosome_content"] = df["mito_kind"].eq("contents")

# -----------------------------
# Save
# -----------------------------
df.to_csv(OUT_PATH, index=False)
print(f"Saved annotated file to: {OUT_PATH}")

# -----------------------------
# Preview
# -----------------------------
preview_cols = [
    c for c in [
        "sample",
        "group_name",
        "source_label_file",
        "label_id",
        "unique_ID",
        "partially_wrapped",
        "partially_wrapped_group_name",
        "mito_kind",
        "corrected_mito_rank",
        "corrected_mito_label_id",
        "corrected_mito_source_label_file",
        "corrected_mito_unique_ID",
        "corrected_mito_group_name",
        "corrected_mito_key",
    ] if c in df.columns
]

display(df[preview_cols].head(30))

Saved annotated file to: c:\Users\narendradp\OneDrive - National Institutes of Health\Documents\Jennifer_Data\2D_measure_NanoSIMS_using_labels\combined_output_before_annotation.csv


,sample,group_name,source_label_file,label_id,unique_ID,partially_wrapped,partially_wrapped_group_name,mito_kind,corrected_mito_rank,corrected_mito_label_id,corrected_mito_source_label_file,corrected_mito_unique_ID,corrected_mito_group_name,corrected_mito_key
0,S18_Ar2,SL_1wk-chase,Mitochondria_S18_Ar2.tif,10001,Mitochondria_S18_Ar2_10001,False,NaN,other,<NA>,<NA>,NaN,NaN,NaN,NaN
1,S18_Ar2,SL_1wk-chase,Mitochondria_S18_Ar2.tif,10003,Mitochondria_S18_Ar2_10003,False,NaN,other,<NA>,<NA>,NaN,NaN,NaN,NaN
2,S18_Ar2,SL_1wk-chase,Mitochondria_S18_Ar2.tif,10005,Mitochondria_S18_Ar2_10005,False,NaN,other,<NA>,<NA>,NaN,NaN,NaN,NaN
3,S18_Ar2,SL_1wk-chase,Mitochondria_S18_Ar2.tif,10015,Mitochondria_S18_Ar2_10015,False,NaN,other,<NA>,<NA>,NaN,NaN,NaN,NaN
4,S18_Ar2,SL_1wk-chase,Mitochondria_S18_Ar2.tif,10021,Mitochondria_S18_Ar2_10021,False,NaN,other,<NA>,<NA>,NaN,NaN,NaN,NaN
5,S18_Ar2,SL_1wk-chase,Mitochondria_S18_Ar2.tif,10026,Mitochondria_S18_Ar2_10026,False,NaN,other,<NA>,<NA>,NaN,NaN,NaN,NaN
6,S18_Ar2,SL_1wk-chase,Mitochondria_S18_Ar2.tif,10027,Mitochondria_S18_Ar2_10027,False,NaN,other,<NA>,<NA>,NaN,NaN,NaN,NaN
7,S18_Ar2,SL_1wk-chase,Mitochondria_S18_Ar2.tif,10028,Mitochondria_S18_Ar2_10028,False,NaN,other,<NA>,<NA>,NaN,NaN,NaN,NaN
8,S18_Ar2,SL_1wk-chase,Mitochondria_S18_Ar2.tif,10029,Mitochondria_S18_Ar2_10029,False,NaN,other,<NA>,<NA>,NaN,NaN,NaN,NaN
9,S18_Ar2,SL_1wk-chase,Mitochondria_S18_Ar2.tif,10030,Mitochondria_S18_Ar2_10030,False,NaN,other,<NA>,<NA>,NaN,NaN,NaN,NaN


In [5]:
# =============================
# EM-only Mitolysosome content rating script
#   - starts at the beginning every time
#   - uses combined_output.csv
#   - uses npz_file for image loading
#   - uses corrected_mito_source_label_file + corrected_mito_label_id for parent lookup
#   - focuses only on Mitolysosome contents
#   - shows:
#       1) parent overview
#       2) parent EM crop
#       3) child EM crop
#   - 4-level morphology criteria
#   - records responses back into df
#   - saves progress after every response
# =============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, clear_output, Javascript
import ipywidgets as widgets

# -----------------------------
# Paths
# -----------------------------
SCRIPT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
CSV_PATH = SCRIPT_DIR / "combined_output.csv"
if not CSV_PATH.exists():
    alt = SCRIPT_DIR / "output" / "combined_output_before_annotation.csv"
    if alt.exists():
        CSV_PATH = alt
    else:
        raise FileNotFoundError("Could not find combined_output.csv in the script directory or output/ subfolder.")

CSV_DIR = CSV_PATH.parent

NPZ_SEARCH_DIRS = [
    CSV_DIR / "output_npz",
    CSV_DIR,
    SCRIPT_DIR / "output_npz",
    SCRIPT_DIR,
    Path("/mnt/data/output_npz"),
    Path("/mnt/data"),
]

OUT_PATH = CSV_DIR / "combined_output_morphology_rated.csv"

# -----------------------------
# Morphology criteria
# -----------------------------
MORPHOLOGY_SCORE_LABELS = {
    1: "apparently undigested",
    2: "partially digested",
    3: "moderately digested",
    4: "severely digested",
}

MORPHOLOGY_SCORE_HELP = (
    "1 – Apparently undigested. All cristae clear and distinguishable, similar to mitochondria outside mitolysosome\n"
    "2 – Partially digested. Some cristae with normal morphology still visible but clear disruption of morphology\n"
    "3 – Moderately digested. No normal cristae, but few remnants that could be cristae, can be very granular\n"
    "4 – Severely digested. No cristae recognizable, very granular"
)

# -----------------------------
# Load dataframe
# -----------------------------
df = pd.read_csv(CSV_PATH).copy()
df["__row_order__"] = np.arange(len(df))

required_cols = ["sample", "source_label_file", "label_id", "npz_file"]
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Expected a '{c}' column in combined_output.csv.")

df["sample"] = df["sample"].astype(str)
df["source_label_file"] = df["source_label_file"].astype(str)
df["label_id"] = pd.to_numeric(df["label_id"], errors="coerce").astype("Int64")
df["npz_file"] = df["npz_file"].astype(str)

# stable key
df["annotation_uid"] = (
    df["sample"].astype(str)
    + "__"
    + df["source_label_file"].astype(str)
    + "__"
    + df["label_id"].astype(str)
)

# -----------------------------
# Focus only on Mitolysosome contents
# -----------------------------
src = df["source_label_file"].astype(str)

if "object_kind" in df.columns and "object_family" in df.columns:
    contents_mask = df["object_kind"].astype(str).eq("contents") & df["object_family"].astype(str).eq("Mitolysosome")
else:
    contents_mask = src.str.match(r"^Mitolysosome\d+-Contents_.+\.tif$", na=False)

# -----------------------------
# Parent lookup from combined_output columns
# -----------------------------
for c in ["corrected_mito_source_label_file", "corrected_mito_label_id", "corrected_mito_unique_ID", "corrected_mito_rank"]:
    if c not in df.columns:
        df[c] = pd.NA

df["corrected_mito_label_id"] = pd.to_numeric(df["corrected_mito_label_id"], errors="coerce").astype("Int64")
df["corrected_mito_rank"] = pd.to_numeric(df["corrected_mito_rank"], errors="coerce").astype("Int64")

# Build a direct parent key using the corrected mapping already present
df["parent_source_label_file"] = df["corrected_mito_source_label_file"]
df["parent_label_id"] = df["corrected_mito_label_id"]
df["parent_annotation_uid"] = pd.NA

uid_lookup = df.set_index("annotation_uid", drop=False)

for idx in df.index[contents_mask]:
    psrc = df.at[idx, "parent_source_label_file"]
    plabel = df.at[idx, "parent_label_id"]
    psample = df.at[idx, "sample"]
    if pd.notna(psrc) and pd.notna(plabel):
        parent_uid = f"{psample}__{psrc}__{int(plabel)}"
        if parent_uid in uid_lookup.index:
            df.at[idx, "parent_annotation_uid"] = parent_uid

# -----------------------------
# Start from the beginning every time
# -----------------------------
for c in [
    "content_morphology_response",
    "content_morphology_score",
    "content_morphology_label",
    "content_morphology_comment",
    "content_morphology_status",
    "content_morphology_rated",
]:
    if c not in df.columns:
        if c == "content_morphology_score":
            df[c] = pd.Series(pd.NA, index=df.index, dtype="Int64")
        elif c == "content_morphology_rated":
            df[c] = False
        else:
            df[c] = pd.Series(pd.NA, index=df.index, dtype="object")

# -----------------------------
# NPZ helpers
# -----------------------------
_npz_cache = {}

def find_npz_path(npz_file):
    p = Path(str(npz_file))
    if p.is_absolute() and p.exists():
        return p
    for root in NPZ_SEARCH_DIRS:
        cand = root / p.name
        if cand.exists():
            return cand
    return None

def load_npz(npz_file):
    if pd.isna(npz_file):
        return None
    key = str(npz_file)
    if key in _npz_cache:
        return _npz_cache[key]
    npz_path = find_npz_path(key)
    if npz_path is None:
        return None
    _npz_cache[key] = np.load(npz_path)
    return _npz_cache[key]

def load_em_and_bbox(npz_file):
    data = load_npz(npz_file)
    if data is None:
        return None, None
    return data["em"], data["bbox_strict"]

def make_overview_em(em, bbox):
    if em is None or bbox is None:
        return None
    y0, y1, x0, x1 = bbox
    full_em = np.zeros((256, 256))
    full_em[y0:y1, x0:x1] = em[:(y1 - y0), :(x1 - x0)]
    return full_em

def show_panel(ax, img, title, full_canvas=False):
    if img is None:
        ax.axis("off")
        ax.text(0.5, 0.5, "Missing NPZ", ha="center", va="center")
        return
    if full_canvas:
        ax.imshow(img, cmap="gray")
        ax.add_patch(
            plt.Rectangle((0, 0), 256, 256, edgecolor="black", facecolor="none", linewidth=2.5, clip_on=False)
        )
    else:
        vmin, vmax = np.percentile(img, 1), np.percentile(img, 99)
        ax.imshow(img, cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")

# -----------------------------
# Hotkeys
# -----------------------------
def install_hotkeys():
    js = r"""
    (function() {
        if (window._morphologyHotkeysInstalled) return;
        window._morphologyHotkeysInstalled = true;

        document.addEventListener('keydown', function(ev) {
            if (!ev || !ev.key) return;
            const target = ev.target;
            if (target && (target.tagName === 'INPUT' || target.tagName === 'TEXTAREA' || target.isContentEditable)) {
                return;
            }

            const map = {
                '1': '1',
                '2': '2',
                '3': '3',
                '4': '4',
                's': 'Skip',
                'S': 'Skip'
            };

            const wanted = map[ev.key];
            if (!wanted) return;

            const buttons = Array.from(document.querySelectorAll('button'));
            const btn = buttons.find(b => (b.innerText || '').trim() === wanted);
            if (btn) {
                ev.preventDefault();
                ev.stopPropagation();
                btn.click();
            }
        }, true);
    })();
    """
    display(Javascript(js))

# -----------------------------
# Annotator
# -----------------------------
class MorphologyAnnotator:
    def __init__(self, df):
        self.df = df
        self.row_by_uid = df.set_index("annotation_uid", drop=False)

        content_df = self.df.loc[contents_mask].copy()
        content_df = content_df.sort_values(["sample", "__row_order__"]).reset_index()

        self.queue = []
        for _, row in content_df.iterrows():
            status = self.df.at[row["index"], "content_morphology_status"]
            if pd.isna(status):
                self.queue.append({
                    "row_index": row["index"],
                    "row": self.df.loc[row["index"]].copy(),
                })

        self.total = int(len(content_df))
        self.done = int(self.df["content_morphology_status"].isin(["rated", "skipped"]).sum())
        self.i = 0

        self.progress = widgets.IntProgress(
            value=self.done,
            min=0,
            max=max(self.total, 1),
            description="Progress:",
            bar_style=""
        )
        self.progress_label = widgets.HTML()
        self.image_out = widgets.Output()
        self.control_out = widgets.Output()
        self.help_out = widgets.Output()

        self.ui = widgets.VBox([
            widgets.HBox([self.progress, self.progress_label]),
            self.help_out,
            self.image_out,
            self.control_out
        ])

    def start(self):
        install_hotkeys()
        display(self.ui)
        self.render_current()

    def save(self):
        self.df.to_csv(OUT_PATH, index=False)

    def update_progress(self):
        self.progress.value = self.done
        self.progress.max = max(self.total, 1)
        self.progress_label.value = f"<b>{self.done} / {self.total}</b> contents completed"

    def render_current(self):
        self.update_progress()

        with self.image_out:
            clear_output(wait=True)

            if self.i >= len(self.queue):
                print("Done. No more Mitolysosome contents to rate.")
                self.save()
                return

            case = self.queue[self.i]
            content_row = case["row"]

            parent_row = None
            parent_uid = content_row.get("parent_annotation_uid", pd.NA)
            if pd.notna(parent_uid) and parent_uid in self.row_by_uid.index:
                parent_row = self.row_by_uid.loc[parent_uid].copy()

            if parent_row is not None:
                parent_em, parent_bbox = load_em_and_bbox(parent_row["npz_file"])
                parent_overview = make_overview_em(parent_em, parent_bbox)
            else:
                parent_em, parent_bbox = None, None
                parent_overview = None

            child_em, child_bbox = load_em_and_bbox(content_row["npz_file"])

            fig, axes = plt.subplots(1, 3, figsize=(18, 6))

            show_panel(
                axes[0],
                parent_overview,
                "Parent overview\n(MISSING)" if parent_row is None else f"Parent overview\n{parent_row['source_label_file']}",
                full_canvas=True
            )
            show_panel(
                axes[1],
                parent_em,
                "Parent EM crop\n(MISSING)" if parent_row is None else f"Parent EM crop\n{parent_row['source_label_file']}",
                full_canvas=False
            )
            show_panel(
                axes[2],
                child_em,
                f"Child EM crop\n{content_row['source_label_file']}",
                full_canvas=False
            )

            parent_title = "MISSING" if parent_row is None else parent_row["source_label_file"]
            fig.suptitle(
                f"Sample: {content_row['sample']} | Parent: {parent_title} | "
                f"Content {self.i + 1} of {len(self.queue)}",
                fontsize=14,
                weight="bold"
            )
            plt.tight_layout(rect=[0, 0, 1, 0.92])
            plt.show()

        self.render_control_panel()

    def render_control_panel(self):
        with self.control_out:
            clear_output(wait=True)

            if self.i >= len(self.queue):
                display(widgets.HTML(value="<b>Annotation complete.</b>"))
                return

            content_row = self.queue[self.i]["row"]

            status = widgets.HTML(
                value=(
                    f"<b>Rate morphology for:</b> {content_row['source_label_file']}<br><br>"
                    f"<b>Criteria:</b><br>"
                    f"{MORPHOLOGY_SCORE_HELP.replace(chr(10), '<br>')}"
                )
            )

            comment = widgets.Text(
                value="",
                placeholder="Optional comment",
                description="Comment:",
                layout=widgets.Layout(width="500px")
            )

            buttons = [
                widgets.Button(description="1", button_style="success", layout=widgets.Layout(width="70px")),
                widgets.Button(description="2", button_style="warning", layout=widgets.Layout(width="70px")),
                widgets.Button(description="3", button_style="danger", layout=widgets.Layout(width="70px")),
                widgets.Button(description="4", button_style="info", layout=widgets.Layout(width="70px")),
                widgets.Button(description="Skip", button_style="", layout=widgets.Layout(width="90px")),
            ]

            def on_click_factory(score_value):
                def _handler(_):
                    self.handle_response(score_value, comment.value.strip())
                return _handler

            for b in buttons:
                if b.description == "Skip":
                    b.on_click(on_click_factory("skip"))
                else:
                    b.on_click(on_click_factory(b.description))

            display(
                widgets.VBox([
                    status,
                    widgets.HBox(buttons),
                    widgets.HTML(value="<i>Tip: use hotkeys 1–4 or s for Skip.</i>"),
                    comment
                ])
            )

    def handle_response(self, score_value, comment_text):
        if self.i >= len(self.queue):
            return

        case = self.queue[self.i]
        row_idx = case["row_index"]

        if score_value == "skip":
            response = "skip"
            score = pd.NA
            label = pd.NA
            status = "skipped"
            rated = False
        else:
            response = str(score_value)
            score = int(score_value)
            label = MORPHOLOGY_SCORE_LABELS[score]
            status = "rated"
            rated = True

        comment = pd.NA if comment_text == "" else comment_text

        self.df.at[row_idx, "content_morphology_response"] = response
        self.df.at[row_idx, "content_morphology_score"] = score
        self.df.at[row_idx, "content_morphology_label"] = label
        self.df.at[row_idx, "content_morphology_comment"] = comment
        self.df.at[row_idx, "content_morphology_status"] = status
        self.df.at[row_idx, "content_morphology_rated"] = rated

        self.done += 1
        self.save()

        self.i += 1
        self.render_current()

# -----------------------------
# Run
# -----------------------------
annotator = MorphologyAnnotator(df)
annotator.start()

# Final save
df.to_csv(OUT_PATH, index=False)
print(f"Saving to: {OUT_PATH}")

<IPython.core.display.Javascript object>

Saving to: c:\Users\narendradp\OneDrive - National Institutes of Health\Documents\Jennifer_Data\2D_measure_NanoSIMS_using_labels\combined_output_morphology_rated.csv
